---
title: "Introduction to GNX-py"
authors: ["Hubert Pierzchala"]
---

# Introduction to GNX-py

This notebook is a guided introduction to GNX-py. We will not start from a high-level black box. Instead, we will first walk through the ingredients of a GNSS positioning pipeline: observations, navigation products, signal transmission time, satellite coordinates, atmospheric corrections, antenna corrections, and the observation equation.

Only after that manual walk-through will we use `SPPConfig` and `SPPSession`, which automate the same chain in the actual library workflow.

# Table of Contents

1. GNX-py as a library
2. Installation for tutorials
3. Sample data and current GNSS signal model
4. Reading observations with `GNSSDataProcessor2`
5. Reading broadcast navigation products and `OrbitData`
6. Broadcast orbit interpolation and satellite clock
7. Signal emission time and Earth rotation/Sagnac correction
8. Elevation, azimuth, and observation filtering
9. Ionosphere, troposphere, relativistic path, PCO, and antenna height corrections
10. Manual SPP observation model and single-epoch least-squares example
11. Automated SPP pipeline with `SPPConfig` and `SPPSession`
12. Interpreting `SPPResult`, residuals, and RMS

# GNX-py library

GNX-py is a GNSS processing library. The package is not only a set of readers; it tries to keep the full scientific workflow visible:

- `gnx_py.spp` contains Single Point Positioning tools. SPP is a good first place to learn the data flow because it uses broadcast products and a relatively small state vector.
- `gnx_py.ppp` contains Precise Point Positioning workflows, including combined and uncombined processing. Advanced modes such as PPP-AR should be treated as experimental unless the corresponding tutorial or validation note says otherwise.
- `gnx_py.ionosphere` contains STEC/VTEC tools, Global Ionosphere Map handling, Klobuchar/NTCM helpers, and explicit bias policy for OSB/DCB/DSB products.
- `gnx_py.orbits` contains SIS/SISRE-style orbit and clock comparison tools.
- `gnx_py.gnss` and `gnx_py.biases` contain shared signal metadata and bias helpers used by higher-level modules.

A useful mental model is: configuration objects describe an experiment, session objects run it, and result objects store tables that can be inspected, plotted, or saved.

## Installation for this tutorial

From a fresh clone, install the package into the environment used by Python, Jupyter, or your IDE:

```bash
python -m pip install -e ".[tutorial]"
```

The `tutorial` extra installs notebook and plotting packages used by the tutorial tree. Core GNX-py runtime dependencies remain in the base package. After installation, `import gnx_py` should work from any directory as long as the same virtual environment is active or selected as the IDE interpreter.

In [ ]:
from __future__ import annotations

import contextlib
import io
import platform
import sys
from dataclasses import fields
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    """Find the GNX repository root without editing Python import paths."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "sample_data").exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root with sample_data.")


REPO_ROOT = find_repo_root()
SAMPLE_DATA = REPO_ROOT / "sample_data"

import gnx_py as gnx

GNX_PATH = Path(gnx.__file__).resolve()

print(f"Python executable: {sys.executable}")
print(f"Python version: {platform.python_version()}")
print(f"gnx_py imported from: {GNX_PATH}")

if REPO_ROOT not in GNX_PATH.parents:
    print("Warning: gnx_py was not imported from this repository checkout.")
    print(f"Reinstall this checkout with: python -m pip install -e {REPO_ROOT}")

## Sample data used here

The tutorial uses compact example input files from `sample_data`. They are kept in the repository as teaching and testing inputs, not as package data installed into `site-packages`.

We will use station `BRUX` on day-of-year 035 of 2024. The code cells use only a ten-minute window so the notebook remains light, but the surrounding explanation follows the full processing logic.

In [ ]:
OBS_PATH = SAMPLE_DATA / "BRUX00BEL_R_20240350000_01D_30S_MO.crx.gz"
NAV_PATH = SAMPLE_DATA / "BRDC00IGS_R_20240350000_01D_MN.rnx"
ATX_PATH = SAMPLE_DATA / "igs20.atx"
DCB_PATH = SAMPLE_DATA / "CAS1OPSRAP_20240350000_01D_01D_DCB.BIA"
GIM_PATH = SAMPLE_DATA / "COD0OPSFIN_20240350000_01D_01H_GIM.INX"
SINEX_PATH = SAMPLE_DATA / "IGS0OPSSNX_20240350000_01D_01D_CRD.SNX"

TUTORIAL_WINDOW = [datetime(2024, 2, 4, 0, 0), datetime(2024, 2, 5, 0, 0)]
NAV_WINDOW = [datetime(2024, 2, 4, 0, 0), datetime(2024, 2, 5, 0, 0)]

path_table = pd.DataFrame(
    [
        ("observation", OBS_PATH),
        ("broadcast navigation", NAV_PATH),
        ("antenna calibration", ATX_PATH),
        ("code biases", DCB_PATH),
        ("ionosphere grid", GIM_PATH),
        ("station coordinates", SINEX_PATH),
    ],
    columns=["role", "path"],
)
path_table["relative path"] = path_table["path"].map(lambda p: str(p.relative_to(REPO_ROOT)))
path_table["exists"] = path_table["path"].map(lambda p: p.exists())
display(path_table[["role", "relative path", "exists"]])

## Current GNSS signal model

A GNSS observable is not just "GPS" or "Galileo". It is tied to a constellation, signal, code observable, phase observable, antenna frequency, and bias convention. GNX-py keeps this mapping in `gnx_py.gnss`.

For BeiDou, the current conservative default is `B1IB3I`. Modern BDS combinations such as `B1CB2a` and `B1CB2b` exist in the metadata, but should be treated as requiring validation unless a specific workflow documents them as ready for production use.

In [ ]:
from gnx_py.gnss import (
    DEFAULT_MODE_BY_SYSTEM,
    SUPPORTED_MODES_BY_SYSTEM,
    SYSTEM_NAMES,
    mode_ionosphere_free_coefficients,
    mode_layout,
    mode_signals,
)

signal_rows = []
for system in ["G", "E", "C"]:
    default_mode = DEFAULT_MODE_BY_SYSTEM[system]
    for mode in SUPPORTED_MODES_BY_SYSTEM[system]:
        layout = mode_layout(mode)
        signal_rows.append(
            {
                "system": system,
                "name": SYSTEM_NAMES[system],
                "mode": mode,
                "default": mode == default_mode,
                "signals": ", ".join(mode_signals(mode)),
                "RINEX code/phase": ", ".join(f"{item['code']}/{item['phase']}" for item in layout),
                "ANTEX": ", ".join(item["antex"] for item in layout),
            }
        )

signal_table = pd.DataFrame(signal_rows)
display(signal_table[signal_table["default"] | signal_table["mode"].isin(["B1CB2a", "B1CB2b"])])

# Example of processing in GNX-py

We now go through the pipeline in the same spirit as a hand calculation. The purpose is educational: we want to see what the later `SPPSession` automates.

The path is:

1. Load observation data and metadata.
2. Load broadcast navigation data.
3. Interpolate satellite coordinates and clocks.
4. Move from reception time to signal emission time.
5. Correct satellite coordinates for Earth rotation during signal travel.
6. Compute elevation/azimuth and apply a simple elevation mask.
7. Inspect ionosphere, troposphere, relativistic, PCO, and antenna-height corrections.
8. Build a simplified SPP observation equation for one epoch.
9. Let `SPPSession` run the production pipeline on the same short data window.

## Reading observations with `GNSSDataProcessor2`

`GNSSDataProcessor2` takes paths to input files and information about the systems and signals we want to use. Here we choose GPS and the `L1L2` mode.

The `use_gfz` argument controls one of the reading paths used for RINEX observation data. GNX-py can use the `georinex` ecosystem for RINEX handling, and it also keeps internal/adapter paths for cases where the observation file or downstream processing needs a different representation. In practice, keep `use_gfz=True` for the current tutorial sample unless you have a reason to test another reader path.

Reference: `georinex` is the Python RINEX reader ecosystem previously used in this tutorial lineage.

In [ ]:
processor = gnx.GNSSDataProcessor2(
    obs_path=OBS_PATH,
    nav_path=NAV_PATH,
    atx_path=ATX_PATH,
    sys={"G"},
    mode="L1L2",
    use_gfz=True,
)

obs_data = processor.load_obs_data(tlim=TUTORIAL_WINDOW)
print(type(obs_data).__name__)

obs_fields = []
for field in fields(obs_data):
    value = getattr(obs_data, field.name)
    obs_fields.append(
        {
            "field": field.name,
            "type": type(value).__name__,
            "shape/size": getattr(value, "shape", len(value) if hasattr(value, "__len__") and not isinstance(value, str) else ""),
        }
    )
display(pd.DataFrame(obs_fields))

We obtained an `ObsData` structure. The most important fields for this introduction are:

- `gps`, `gal`, `bds`: observation tables for active constellations. In this first example only `gps` is populated.
- `sat_pco`: satellite phase-center offsets read from ANTEX.
- `rec_pco`: receiver antenna phase-center offsets read from ANTEX.
- `meta`: station metadata read from the RINEX header, including station name, antenna, approximate coordinates, geodetic coordinates, observation interval, and available observation types.

This is one of the central design ideas of GNX-py: raw files are converted into structured objects before scientific processing starts.

In [ ]:
gps_obs = obs_data.gps.copy()

print(f"GPS observation table shape: {gps_obs.shape}")
display(gps_obs.reset_index().head(6))

meta_names = [
    "station", "antenna", "radome", "antenna_delta_hen_m", "approx_xyz_m",
    "approx_flh_deg_m", "interval_s", "gps_obs_types", "gal_obs_types",
    "first_epoch", "last_epoch", "phase_shift",
]
meta_table = pd.DataFrame(
    {
        "field": meta_names,
        "value": [repr(value) for value in obs_data.meta],
    }
)
display(meta_table.head(8))

## Antenna metadata: satellite and receiver PCO

Another important part of `ObsData` is antenna metadata. Phase-center offsets (PCO) describe the vector from the antenna reference point to the effective phase center for a given signal/frequency.

GNX-py stores satellite PCO in `sat_pco`, usually keyed by satellite and ANTEX frequency code, and receiver PCO in `rec_pco`, keyed by antenna, radome, and signal frequency. These values later enter the observation model through line-of-sight projections.

In [ ]:
sat_pco_sample = pd.DataFrame(
    [
        {"key": str(key), "pco_mm": value}
        for key, value in list(obs_data.sat_pco.items())[:6]
    ]
)
rec_pco_sample = pd.DataFrame(
    [
        {"key": str(key), "pco_mm": value}
        for key, value in list(obs_data.rec_pco.items())[:6]
    ]
)

print("Satellite PCO sample")
display(sat_pco_sample)
print("Receiver PCO sample")
display(rec_pco_sample)

## Reading broadcast navigation products

Now we load broadcast navigation data with `load_broadcast_orbit()`. The returned `OrbitData` object may contain GPS, Galileo, and BeiDou navigation tables as well as broadcast ionosphere coefficients.

For GPS, the navigation rows contain Keplerian orbital elements and satellite clock parameters. They are not yet satellite coordinates at observation epochs. We still need to select the proper navigation message and interpolate/propagate the satellite state.

In [ ]:
nav_data = processor.load_broadcast_orbit(tlim=NAV_WINDOW)
print(type(nav_data).__name__)

orbit_fields = []
for field in fields(nav_data):
    value = getattr(nav_data, field.name)
    orbit_fields.append(
        {
            "field": field.name,
            "type": type(value).__name__,
            "shape/size": getattr(value, "shape", len(value) if hasattr(value, "__len__") and not isinstance(value, str) else ""),
        }
    )
display(pd.DataFrame(orbit_fields))

gps_orb = nav_data.gps_orb.copy()
print(f"GPS broadcast navigation table shape: {gps_orb.shape}")
display(gps_orb.reset_index().head(6))

## Broadcast coordinates interpolation algorithm

The GPS broadcast message contains parameters of the satellite orbit. To obtain satellite ECEF coordinates at epoch `t`, the algorithm follows the classical broadcast ephemeris sequence:

$$\begin{aligned}
t_k &= t - t_{oe}\quad(\text{apply GPS week crossover}) \\
M_k &= M_0 + \left(\sqrt{\mu/a^3}+\Delta n\right)t_k \\
\text{solve }& M_k = E_k - e\sin E_k \\
v_k &= 2\tan^{-1}\!\left(\sqrt{\frac{1+e}{1-e}}\tan\frac{E_k}{2}\right) \\
u_k &= \omega + v_k + c_{uc}\cos 2(\omega{+}v_k) + c_{us}\sin 2(\omega{+}v_k) \\
r_k &= a(1 - e\cos E_k) + c_{rc}\cos 2(\omega{+}v_k) + c_{rs}\sin 2(\omega{+}v_k) \\
i_k &= i_0 + \dot{i}\,t_k + c_{ic}\cos 2(\omega{+}v_k) + c_{is}\sin 2(\omega{+}v_k) \\
\lambda_k &= \Omega_0 + (\dot{\Omega}-\omega_E)t_k - \omega_E t_{oe} \\
\begin{bmatrix}X\\Y\\Z\end{bmatrix}
&= \mathbf{R}_3(-\lambda_k)\,\mathbf{R}_1(-i_k)\,\mathbf{R}_3(-u_k)\,\begin{bmatrix}r_k\\0\\0\end{bmatrix}
\end{aligned}$$

The satellite clock is modeled with broadcast clock coefficients and the relativistic satellite-clock term:

$$\begin{aligned}
dt^{sat} &= a_0 + a_1 (t-t_0) + a_2 (t-t_0)^2 + \Delta t_{rel}\\
\Delta t_{rel} &= -2 \cdot \frac{r_{sat} \cdot v_{sat}}{c^2}
\end{aligned}$$

Where:

$$\begin{align*}
t_{0} &: \text{satellite clock reference epoch} \\
t_{oe} &: \text{ephemeris reference epoch} \\
t_k &= t - t_{oe}\ \text{with week crossover handling} \\
M_0 &: \text{mean anomaly at }t_{oe} \\
\mu &: \text{Earth gravitational constant} \\
E_k &: \text{eccentric anomaly solving } M_k=E_k-e\sin E_k \\
v_k &: \text{true anomaly} \\
c_{uc},c_{us} &: \text{corrections to argument of latitude} \\
c_{rc},c_{rs} &: \text{corrections to orbital radius} \\
c_{ic},c_{is} &: \text{corrections to inclination} \\
\Omega_0 &: \text{longitude of ascending node at }t_{oe} \\
\omega_E &: \text{Earth rotation rate} \\
r_{sat} &: \text{satellite position vector} \\
v_{sat} &: \text{satellite velocity vector}
\end{align*}$$

GNX-py wraps this broadcast propagation in `BroadcastInterp`. The class selects a valid navigation message for each satellite and observation epoch, propagates the state, and attaches satellite coordinates and clocks to the observation table.

## Signal emission time

A receiver timestamp is the signal reception time. Satellite coordinates, however, should refer to the signal emission time. A first approximation uses the measured pseudorange:

$$\begin{aligned}
R &= c\big(t_{\rm rcv}[{\rm reception}] - t^{sat}[{\rm emission}]\big) \\
t^{sat}[{\rm emission}] &= t_{\rm rcv}[{\rm reception}] - R/c \\
T[{\rm emission}] &= t_{\rm rcv}[{\rm reception}] - R/c - dt^{sat}
\end{aligned}$$

Here `R` is the measured pseudorange, `dt^{sat}` is the satellite clock offset, and `c` is the speed of light. In practice, GNX-py first propagates the satellite to reception time, estimates emission time, and then propagates again at emission time.

In [ ]:
# Build an ionosphere-free GPS code combination for the educational manual path.
manual_obs = gps_obs.copy()
gnx.make_ionofree(manual_obs, mode="L1L2", sys="G")

interp = gnx.BroadcastInterp(
    obs=manual_obs,
    nav=gps_orb,
    sys="G",
    mode="L1L2",
    emission_time=True,
)

with contextlib.redirect_stdout(io.StringIO()) as interp_log:
    obs_with_sat = interp.interpolate()

for line in interp_log.getvalue().splitlines()[:3]:
    print(line)

signal_travel_time = obs_with_sat["P3"] / 299_792_458.0 + obs_with_sat["clk"]
obs_with_sat["emission_epoch_est"] = (
    obs_with_sat.index.get_level_values("time") - pd.to_timedelta(signal_travel_time, unit="s")
)

print(f"Observation rows with selected broadcast coordinates: {obs_with_sat.shape[0]}")
display(obs_with_sat[["P3", "x", "y", "z", "clk", "emission_epoch_est"]].reset_index().head(6))

The table now contains observation columns (`C1W`, `C2W`, `P3`, ...), selected broadcast message parameters, satellite coordinates (`x`, `y`, `z`), satellite clock `clk`, and the estimated emission epoch.

The coordinates are still ECEF coordinates tied to the Earth orientation at the relevant epoch. Because Earth rotates while the signal travels, the next step is the Sagnac/Earth-rotation correction.

## Earth rotation / Sagnac correction

During signal travel the Earth rotates. If this is ignored, the line-of-sight geometry is slightly inconsistent: the satellite coordinate is tied to Earth orientation at emission time, while the receiver is evaluated at reception time.

GNX-py applies the correction as a rotation around the Earth rotation axis:

$$\begin{aligned}
\tilde{\mathbf r}^{\,sat} &\equiv \text{satellite ECEF at emission time} \\
\Delta t &= \frac{\left\|\tilde{\mathbf r}^{\,sat}-\mathbf r_{rcv}\right\|}{c} \\
\mathbf r^{sat} &= \mathbf{R}_3\!\big(\omega_E\,\Delta t\big)\,\tilde{\mathbf r}^{\,sat},\quad
\mathbf{R}_3(\theta)=\begin{bmatrix}\cos\theta&\sin\theta&0\\-\sin\theta&\cos\theta&0\\0&0&1\end{bmatrix}
\end{aligned}$$

The same wrapper also computes azimuth and elevation from receiver coordinates.

In [ ]:
obs_geom = gnx.CustomWrapper(
    obs=obs_with_sat,
    flh=obs_data.meta[5],
    xyz_a=obs_data.meta[4],
    mode="L1L2",
    epochs=None,
).run()

sagnac_delta = (
    obs_geom[["xe", "ye", "ze"]].to_numpy()
    - obs_geom[["xe_ur", "ye_ur", "ze_ur"]].to_numpy()
)
sagnac_norm = np.linalg.norm(sagnac_delta, axis=1)

print(f"Median coordinate rotation magnitude: {np.median(sagnac_norm):.3f} m")
display(obs_geom[["x", "y", "z", "xe", "ye", "ze", "ev", "az"]].reset_index().head(6))

## Elevation, azimuth, and a simple observation mask

Elevation and azimuth give the local viewing geometry. Low-elevation observations usually have larger atmospheric errors, multipath, and modeling uncertainty. A simple SPP workflow commonly applies an elevation mask before solving the receiver position.

In [ ]:
manual_pipeline = obs_geom[obs_geom["ev"] > 10].copy()

print(f"Rows before elevation mask: {len(obs_geom)}")
print(f"Rows after 10 degree mask: {len(manual_pipeline)}")

elevation_summary = manual_pipeline.reset_index().groupby("sv")["ev"].agg(["min", "median", "max"])
display(elevation_summary.head(10))

## Klobuchar ionosphere model

For single-frequency code observations the ionosphere is one of the dominant error sources. The GPS broadcast message carries Klobuchar coefficients. The model estimates a slant group delay using the receiver position, satellite azimuth/elevation, time of week, and broadcast coefficients.

Inputs in semicircles unless noted; `E` is elevation and `A` is azimuth:

$$\begin{aligned}
\psi &= \frac{0.0137}{E+0.11}-0.022 && \text{Earth-centered angle} \\
\phi_I &= \phi_u + \psi\cos A,\quad \phi_I\in[-0.416,\,0.416] && \text{IPP latitude} \\
\lambda_I &= \lambda_u + \frac{\psi\sin A}{\cos\phi_I} && \text{IPP longitude} \\
\phi_m &= \phi_I + 0.064\cos(\lambda_I-1.617) && \text{geomagnetic latitude at IPP} \\
t' &= \big(43200\,\lambda_I + t_{\rm GPS}\big)\bmod 86400 && \text{local time} \\
A_I &= \sum_{n=0}^3 \alpha_n \phi_m^n,\quad A_I\ge 0 && \text{amplitude} \\
P_I &= \sum_{n=0}^3 \beta_n \phi_m^n,\quad P_I\ge 72000 && \text{period} \\
X_I &= \frac{2\pi\,(t'-50400)}{P_I} && \text{phase} \\
F &= 1 + 16(0.53 - E)^3 && \text{slant factor} \\
I_{L1} &=
\begin{cases}
\big[5{\cdot}10^{-9} + A_I\big(1-\tfrac{X_I^2}{2}+\tfrac{X_I^4}{24}\big)\big]F, & |X_I|\le 1.57,\\[0.5ex]
5{\cdot}10^{-9}F, & |X_I|>1.57,
\end{cases} \\
\Delta R &= c\,I_{L1},\qquad I_f = \left(\frac{f_{L1}}{f}\right)^2 I_{L1}
\end{aligned}$$

In this notebook the final manual solution uses the ionosphere-free `L1L2` combination, so the Klobuchar values are shown mainly to explain what the model contributes in single-frequency SPP.

In [ ]:
from gnx_py.ionosphere.models import klobuchar

manual_reset = manual_pipeline.reset_index()
receiver_lat, receiver_lon, receiver_h = obs_data.meta[5]

manual_pipeline["ion_l1_m"] = [
    klobuchar(
        azimuth=row.az,
        elev=row.ev,
        fi=receiver_lat,
        lambda_=receiver_lon,
        tow=row.time.hour * 3600 + row.time.minute * 60 + row.time.second,
        beta=nav_data.gpsb,
        alfa=nav_data.gpsa,
    )
    for row in manual_reset.itertuples(index=False)
]

display(manual_pipeline[["ev", "az", "ion_l1_m"]].reset_index().head(6))

## Troposphere models: Saastamoinen and Collins

Tropospheric delay is non-dispersive at GNSS frequencies, so an ionosphere-free code combination does not remove it. GNX-py exposes several troposphere choices in configuration objects. In this introduction we use Saastamoinen, while other workflows may use Niell/Collins-style modeling depending on the processing mode.

For Saastamoinen, the tutorial uses the standard-atmosphere version implemented in the library:

$$
\begin{aligned}
\epsilon &= |\mathrm{ev}|\cdot \frac{\pi}{180} \\
P(h) &= Pr\left(1 - 2.26\times 10^{-5}h\right)^{5.225} \\
T(h) &= Tr - 0.0065\,h \\
H(h) &= Hr\,\exp\!\left(-6.396\times 10^{-4}h\right) \\
e(h) &= 0.01\,H(h)\,
\exp\!\big( -37.2465 + 0.213166\,T(h) - 2.56908{\times}10^{-4}T(h)^2 \big)
\end{aligned}
$$

$$\begin{aligned}
d_{\text{trop}}(\epsilon,h)
&= \frac{0.002277}{\sin\epsilon}\!\left[P(h) - \frac{B(h)}{\tan^2\epsilon}\right]
+ \frac{0.002277}{\sin\epsilon}\!\left(\frac{1255}{T(h)} + 0.05\right)e(h)
\end{aligned}$$

In [ ]:
from gnx_py.troposphere import saastamoinen

manual_pipeline["tro"] = saastamoinen(manual_pipeline["ev"].to_numpy(), receiver_h)

display(manual_pipeline[["ev", "tro"]].reset_index().head(6))
print(f"Troposphere delay range in this window: {manual_pipeline['tro'].min():.3f} to {manual_pipeline['tro'].max():.3f} m")

## Relativistic path range correction

There are two relativistic effects in this part of the pipeline. The broadcast satellite clock already includes a relativistic satellite-clock term. A separate path-range correction can also be applied to the geometric range:

$$\begin{aligned}
\rho_{\text{geom}} &= \left\| \mathbf r^{sat} - \mathbf r_{rcv} \right\|, \\
\Delta \rho_{\text{rel}} &= \frac{2\,\mu}{c^2}\,\ln\!\left(\frac{r^{sat}+r_{rcv}+\rho_{\text{geom}}}{r^{sat}+r_{rcv}-\rho_{\text{geom}}}\right), \\
\rho_{\text{corr}} &= \rho_{\text{geom}} + \Delta \rho_{\text{rel}} .
\end{aligned}$$

Here `r^{sat}` and `r_{rcv}` are vector norms, `c` is the speed of light, and `mu` is Earth's gravitational constant. The magnitude is small in SPP, but it is useful to see it because PPP workflows keep such terms explicit.

In [ ]:
MU_EARTH = 3.986004418e14
CLIGHT = 299_792_458.0

receiver_xyz = obs_data.meta[4].astype(float)
sat_xyz = manual_pipeline[["xe", "ye", "ze"]].to_numpy(dtype=float)
geom_range = np.linalg.norm(sat_xyz - receiver_xyz, axis=1)
sat_norm = np.linalg.norm(sat_xyz, axis=1)
receiver_norm = np.linalg.norm(receiver_xyz)

manual_pipeline["dprel"] = (2 * MU_EARTH / CLIGHT**2) * np.log(
    (sat_norm + receiver_norm + geom_range) / (sat_norm + receiver_norm - geom_range)
)

display(manual_pipeline[["ev", "dprel"]].reset_index().head(6))
print(f"Relativistic path correction range: {manual_pipeline['dprel'].min():.4f} to {manual_pipeline['dprel'].max():.4f} m")

## Receiver PCO and antenna height as line-of-sight corrections

The receiver PCO in ANTEX is given in a local topocentric frame. To use it in the observation model, GNX-py converts the local vector to ECEF and projects it onto the line of sight.

For a satellite unit line-of-sight vector `e`, a simplified receiver PCO projection is:

$$\Delta_{PCO,f}^{rcv} = \mathbf e^T \mathbf p_{f}^{ECEF}$$

For an ionosphere-free combination, the frequency-dependent receiver PCO is combined with the same coefficients as the code observations:

$$\Delta_{PCO,IF}^{rcv} = k_1\Delta_{PCO,1}^{rcv} - k_2\Delta_{PCO,2}^{rcv}$$

The antenna height from the RINEX header can be treated similarly: it is a local displacement of the antenna reference point and can be projected onto the line of sight. Satellite PCO follows the same geometric idea, but uses satellite body-frame information and Sun/satellite attitude handling inside the processing pipeline.

In [ ]:
def receiver_pco_m(signal_code: str) -> np.ndarray:
    antenna_name, radome = obs_data.meta[1], obs_data.meta[2]
    for key, value in obs_data.rec_pco.items():
        if key == (antenna_name, radome, signal_code):
            return np.array(value, dtype=float) / 1000.0
    return np.zeros(3)

los_to_receiver = -(manual_pipeline[["xe", "ye", "ze"]].to_numpy(dtype=float) - receiver_xyz)
los_unit = los_to_receiver / np.linalg.norm(los_to_receiver, axis=1)[:, None]

pco_l1_ecef = gnx.enu_to_ecef(dENU=receiver_pco_m("G01"), flh=obs_data.meta[5])
pco_l2_ecef = gnx.enu_to_ecef(dENU=receiver_pco_m("G02"), flh=obs_data.meta[5])

manual_pipeline["pco_los_l1"] = los_unit @ pco_l1_ecef
manual_pipeline["pco_los_l2"] = los_unit @ pco_l2_ecef
k1, k2 = mode_ionosphere_free_coefficients("L1L2")
manual_pipeline["pco_los_l3"] = k1 * manual_pipeline["pco_los_l1"] - k2 * manual_pipeline["pco_los_l2"]

# RINEX stores antenna delta as H/E/N. Convert to an E/N/U displacement before ECEF projection.
ant_h, ant_e, ant_n = obs_data.meta[3]
ant_height_ecef = gnx.enu_to_ecef(dENU=np.array([ant_e, ant_n, ant_h]), flh=obs_data.meta[5])
manual_pipeline["antenna_h_los"] = los_unit @ ant_height_ecef

display(manual_pipeline[["pco_los_l1", "pco_los_l2", "pco_los_l3", "antenna_h_los"]].reset_index().head(6))

## Manual SPP observation model

At this point we have enough ingredients to write a simplified SPP code observation equation. For an ionosphere-free GPS code combination in this short educational example:

$$\begin{aligned}
P_{IF}^{corr} &= P_{IF} + c\,dt^{sat} - T - \Delta\rho_{rel} + \Delta_{PCO,IF}^{rcv} \\
P_{IF}^{corr} &\approx \rho(\mathbf r_{rcv}, \mathbf r^{sat}) + b_{rcv}
\end{aligned}$$

The unknowns are the receiver position correction and receiver clock bias. In matrix form for one epoch:

$$\mathbf v = \mathbf A\,\Delta\mathbf x$$

where each row of `A` contains the line-of-sight partial derivatives and a clock column. The next cell solves one epoch from the prepared observation table. This is not meant to replace `SPPSession`; it is a transparent miniature of what the automated pipeline later does for every epoch.

In [ ]:
first_epoch = manual_pipeline.index.get_level_values("time").min()
epoch_obs = manual_pipeline.xs(first_epoch, level="time").copy()

x_est = receiver_xyz.copy()
clock_est_m = 0.0

P_corr = (
    epoch_obs["P3"].to_numpy(dtype=float)
    + CLIGHT * epoch_obs["clk"].to_numpy(dtype=float)
    - epoch_obs["tro"].to_numpy(dtype=float)
    - epoch_obs["dprel"].to_numpy(dtype=float)
    + epoch_obs["pco_los_l3"].to_numpy(dtype=float)
)
sat_epoch_xyz = epoch_obs[["xe", "ye", "ze"]].to_numpy(dtype=float)

iteration_rows = []
for iteration in range(6):
    diff = sat_epoch_xyz - x_est
    rho = np.linalg.norm(diff, axis=1)
    design = np.column_stack((-diff / rho[:, None], np.ones(len(rho))))
    residual = P_corr - (rho + clock_est_m)
    step = np.linalg.lstsq(design, residual, rcond=None)[0]
    x_est = x_est + step[:3]
    clock_est_m = clock_est_m + step[3]
    iteration_rows.append(
        {
            "iteration": iteration + 1,
            "dX_m": step[0],
            "dY_m": step[1],
            "dZ_m": step[2],
            "dClock_m": step[3],
            "position_step_norm_m": np.linalg.norm(step[:3]),
        }
    )

manual_enu = gnx.ecef_to_enu(dXYZ=x_est - receiver_xyz, flh=obs_data.meta[5])
manual_solution = pd.Series(
    {
        "epoch": first_epoch,
        "satellites": len(epoch_obs),
        "east_error_m": manual_enu[0],
        "north_error_m": manual_enu[1],
        "up_error_m": manual_enu[2],
        "receiver_clock_m": clock_est_m,
    }
)

display(pd.DataFrame(iteration_rows))
display(manual_solution.to_frame("value"))

The manual solution shows the idea, not a production-quality positioning engine. It uses one epoch, a compact correction set, and a very simple least-squares loop. The important point is the structure: after satellite coordinates and corrections are prepared, SPP reduces to geometry plus receiver clock estimation.

Now we can switch to the GNX-py session API. The session performs the same chain repeatedly, handles configuration choices, stores residuals, and returns a structured result object.

# SPP Pipeline in GNX-py

The manual preparation above is useful for learning, but it is not how we want users to run routine processing. In GNX-py, the pipeline is represented by:

- `SPPConfig`: input files, systems, frequency modes, correction models, masks, and solver options,
- `SPPSession`: the object that reads the data, applies preprocessing and corrections, runs the solver, and returns `SPPResult`.

This separation is deliberate. Configuration makes scientific assumptions visible; the session keeps the workflow reproducible.

In [ ]:
spp_config = gnx.SPPConfig(
    obs_path=OBS_PATH,
    nav_path=NAV_PATH,
    atx_path=ATX_PATH,
    dcb_path=DCB_PATH,
    gim_path=GIM_PATH,
    sinex_path=SINEX_PATH,
    sys={"G"},
    gps_freq="L1L2",
    gal_freq="E1E5a",
    bds_freq="B1IB3I",
    orbit_type="broadcast",
    ionosphere_model="klobuchar",
    troposphere_model="saastamoinen",
    time_limit=TUTORIAL_WINDOW,
    day_of_year=35,
    station_name="BRUX",
    ev_mask=10,
    use_gfz=True,
    sat_pco="los",
    rec_pco=True,
    antenna_h=True,
    rel_path=True,
    screen=False,
)

config_summary = pd.Series(
    {
        "systems": ", ".join(sorted(spp_config.sys)),
        "GPS frequency mode": spp_config.gps_freq,
        "Galileo frequency mode": spp_config.gal_freq,
        "BeiDou frequency mode": spp_config.bds_freq,
        "orbit source": spp_config.orbit_type,
        "ionosphere model": spp_config.ionosphere_model,
        "troposphere model": spp_config.troposphere_model,
        "satellite PCO": spp_config.sat_pco,
        "receiver PCO": spp_config.rec_pco,
        "antenna height": spp_config.antenna_h,
        "relativistic path": spp_config.rel_path,
        "station": spp_config.station_name,
        "time window": f"{TUTORIAL_WINDOW[0]} to {TUTORIAL_WINDOW[1]}",
    }
)
display(config_summary.to_frame("value"))

In [ ]:
with contextlib.redirect_stdout(io.StringIO()) as spp_log:
    spp_result = gnx.SPPSession(spp_config).run()

for line in spp_log.getvalue().splitlines()[:5]:
    print(line)

solution = spp_result.solution.copy()
print(f"Solved epochs: {len(solution)}")
display(solution[["time", "de", "dn", "du", "n_gps", "rms_v", "iters"]].head())
display(solution[["time", "de", "dn", "du", "n_gps", "rms_v", "iters"]].tail())

## `SPPResult`: solution, residuals, and diagnostics

The result is a dataclass. The most important field is `solution`, which stores estimated receiver coordinates, receiver clock terms, quality diagnostics, and ENU errors when a reference coordinate is available.

The residual tables are split by constellation:

- `residuals_gps`
- `residuals_gal`
- `residuals_bds`

In this GPS-only example only the GPS residual table is populated. Later tutorials will use this structure to compare systems and inspect processing diagnostics.

In [ ]:
result_fields = []
for field in fields(spp_result):
    value = getattr(spp_result, field.name)
    result_fields.append(
        {
            "field": field.name,
            "type": type(value).__name__,
            "shape": getattr(value, "shape", None),
        }
    )

display(pd.DataFrame(result_fields))

print(type(spp_result.residuals_gps).__name__, spp_result.residuals_gps.shape)
print("Galileo residuals:", spp_result.residuals_gal)
print("BeiDou residuals:", spp_result.residuals_bds)
display(spp_result.residuals_gps.reset_index().head(6))

## Interpreting position errors and RMS

The columns `de`, `dn`, and `du` are local East, North, and Up errors with respect to the reference station coordinates. A compact way to summarize the run is to compute component RMS values and a 3D RMS:

$$\mathrm{RMS}_e = \sqrt{\frac{1}{N}\sum_i e_i^2},\quad
\mathrm{RMS}_n = \sqrt{\frac{1}{N}\sum_i n_i^2},\quad
\mathrm{RMS}_u = \sqrt{\frac{1}{N}\sum_i u_i^2}$$

$$\mathrm{RMS}_{3D}=\sqrt{\mathrm{RMS}_e^2+\mathrm{RMS}_n^2+\mathrm{RMS}_u^2}$$

For this notebook the numbers are not a benchmark. They are a quick check that the session produced a physically interpretable positioning time series.

In [ ]:
enu_cols = ["de", "dn", "du"]
rms_components = np.sqrt((solution[enu_cols] ** 2).mean())
rms_table = rms_components.rename({"de": "east", "dn": "north", "du": "up"}).to_frame("RMS_m")
rms_table.loc["3D", "RMS_m"] = float(np.sqrt((rms_components ** 2).sum()))
display(rms_table)

ax = solution.set_index("time")[enu_cols].plot(marker="o", linewidth=0.5,markersize=0.3,subplots=True,figsize=(10,8))
ax[1].set_ylabel("position error [m]")
ax[-1].set_xlabel("epoch")
ax[0].set_title("SPP session: BRUX, GPS L1/L2, broadcast navigation")
for a in ax:
    a.grid(True, alpha=0.3)
plt.tight_layout()

# Summary

In this chapter you saw the processing chain from raw input files to a structured SPP result:

1. `GNSSDataProcessor2` loaded observations, antenna metadata, station metadata, and broadcast navigation products.
2. `BroadcastInterp` selected broadcast messages and propagated satellite coordinates and clocks.
3. We moved from reception time to emission time and corrected satellite coordinates for Earth rotation.
4. We inspected elevation/azimuth, Klobuchar ionosphere, Saastamoinen troposphere, relativistic path range, receiver/satellite PCO ideas, and antenna height projection.
5. We solved a small manual one-epoch SPP example to see the geometry of the problem.
6. We then used `SPPConfig` and `SPPSession` to run the automated GNX-py pipeline.

The next tutorial should build on this foundation and move from SPP to PPP, where precise products, combined/uncombined observables, ionospheric constraints, and bias policy become more central.